In [6]:
# !pip install lightrag-hku

In [8]:
from IPython.display import Image

- 快速理解/上手 agent 项目
    - 基本概念 / workflow (pipeline): 如果有论文读论文，或者读项目框架介绍；
    - prompts：$f(x, \text{prompt})$，基于 langfuse （本地部署，监控 api 输入输出）
        - llm-based workflow：基本上所有的模块都是 llm 扮演的；
        - 理解了 prompt，就约等于理解各个环节的输入、输出以及处理过程；
- 今天我们介绍 lightrag，很快我们全面地介绍 paper2slides

### core concepts

> 基于图的文本索引（Graph-based Text Indexing） 和 双层检索范式（Dual-level Retrieval Paradigm）。
- 基于图的文本索引 (Graph-based Text Indexing)
$$\text{Graph Index} = \text{Dedupe}(\underbrace{\text{Prof}(\underbrace{\text{Extract}(\text{Text})}_{\text{结构化}})}_{\text{语义化}})$$
    - 实体与关系抽取 (Entity & Rel Extraction - $R(\cdot)$): llm-based 
    - 去重 (Deduplication - $D(\cdot)$)
    - LLM 画像/分析 (LLM Profiling - $P(\cdot)$)
- 双层检索范式 (Dual-level Retrieval Paradigm)
    - 查询处理：query + llm => low-level keys，high-level keys
    - 低层级关键词提取 (Low-level Keys)
        - 具体的实体（Specific Entities）。
    - 高层级关键词提取 (High-level Keys)
        - 宏观的主题或概念（Broader Topics）。
    - 检索内容 (Retrieved Content)
        - 系统利用上述两类关键词，在索引图中进行检索，获取三种类型的信息：
            - Entities（实体）：相关的具体节点信息。
            - Relations（关系）：节点之间的连接路径和关系描述。
            - Original Text（原始文本）：与这些实体和关系关联的原始文本片段。
    - 生成上下文与回答 (Contexts & Answer)
- lightrag vs. graphrag
    - 增量更新（incremental update）：对于新文档 $\mathcal D'$，lightrag 不需要重建整个图。它对 $\mathcal D'$ 执行相同的提取步骤得到 $\hat{\mathcal D}'$，然后通过集合并集操作 $\hat{\mathcal V}\cup \hat{\mathcal V}'$ 和 $\hat{\mathcal E}\cup \hat{\mathcal E}'$，快速合并。这解决了 GraphRAG 需要重新进行社区发现（Clustering，community detection）的高耗时痛点。

In [10]:
Image(url='../figs/lightrag.png', width=600)

### prompts

- https://github.com/HKUDS/LightRAG/blob/main/lightrag/prompt.py
    - **entity_extraction_system_prompt、entity_extraction_user_prompt**: first query 
        - entity_extraction_examples
    - **entity_continue_extraction_user_prompt**: self-correction (两轮对话)
        - Based on the last extraction task, identify and extract any **missed or incorrectly formatted** entities and relationships from the input text.
    - summarize_entity_descriptions
    - keywords_extraction: Your purpose is to identify both high-level and low-level keywords in the user's query that will be used for effective document retrieval.
        - keywords_extraction_examples
    - rag_response: final response

### core implementation

- core methods
    - (a)insert
    - (a)query
- 中间结果文件 store: `kv_storage`/`vector_storage`/`graph_storage`
    - `graph_chunk_entity_relation.graphml`
    - `kv`: 原始数据仓库 (kv_store_*.json)
        - kv_store_full_docs.json:
        - kv_store_text_chunks.json:
            - 原始文本太长，LightRAG 会把它切成小块（Chunks）。
            - 这是 RAG 检索的基本单位。
        - kv_store_full_entities.json & kv_store_full_relations.json:
    - `vdb`: LightRAG 把文字变成了向量（Embedding）。默认使用 NanoVectorDB（纯本地 JSON 实现）。
        - vdb_entities.json: 实体的向量索引。
            - 当你搜 "Who likes graphs?"，系统会把这句话变成向量，然后去这里找最接近的实体（比如 "Alice"）。
        - vdb_relationships.json: 关系的向量索引。
        - vdb_chunks.json: 文本块的向量索引（传统 RAG 的做法）。
- query 当你调用 await rag.aquery("Who acquired FooBar?") 时，系统是这样使用这些文件的：
    - 关键词匹配/向量搜索: 先去 vdb_entities.json 找 "FooBar" 和 "Acquired" 相关的实体。
    - 图遍历: 在 graph_chunk_entity_relation.graphml 中找到这些实体节点，看看它们连着谁（发现 "ExampleCorp"）。
    - 补充细节: 去 kv_store_text_chunks.json 拿出相关的原始文本块，作为上下文（Context）。
    - 生成答案: 把这些信息喂给 GPT，生成最终回复。
- lightrag 默认启用 LLM Response Cache
    - `kv_store_llm_response_cache.json`

### langfuse

- LightRAG 原生集成了 Langfuse。只要检测到环境变量，它会自动替换 OpenAI Client 为 Langfuse 的 Client，从而在后台记录完整的 Trace。
- 只需要在 `.env` 或环境变量中设置：

```
LANGFUSE_PUBLIC_KEY=pk-lf-...
LANGFUSE_SECRET_KEY=sk-lf-...
LANGFUSE_HOST="http://localhost:3000"
```

- lightrag 日志监控: `INFO: Langfuse observability enabled for OpenAI client`